# ALUAR S.A.I.C. (ALUA.BA) — Notebook Maestro Canónico (Módulos M1–M13)
**Valuación Institucional Cuantitativa & Estocástica**
*Cátedra de Economía y Técnica Bursátil — FCE UNCuyo*
*Analista:* Federico Agustín Chillón

Este notebook contiene el pipeline completo, secuencial y autónomo de los 13 módulos del modelo de valuación oficial.

### Módulo 1: Ingesta de Datos y Descarga de Mercado (M1)
Carga y procesamiento de datos históricos de cotización de ALUA.BA, índice Merval, S&P 500, LME Aluminio, Tipo de Cambio CCL y Riesgo País (EMBI+).

In [ ]:
import numpy as np
import pandas as pd
import datetime

print("=== MÓDULO 1: INGESTA DE DATOS DE MERCADO ===")
# Parámetros oficializados del modelo de mercado al 23-Jul-2026
spot_alua = 982.50
ccl_spot = 1584.25
rf_us_10y = 0.0470
embi_ar_pb = 441.0
embi_ar_pct = 0.0441
lme_aluminio_usd = 2450.0

print(f"✓ Spot ALUA.BA        : ARS {spot_alua:,.2f}")
print(f"✓ Tipo Cambio CCL     : ARS {ccl_spot:,.2f}")
print(f"✓ Tasa Libre Riesgo Rf: {rf_us_10y:.2%}")
print(f"✓ EMBI+ Argentina     : {embi_ar_pb:.0f} pb ({embi_ar_pct:.2%})")
print(f"✓ LME Aluminio        : USD {lme_aluminio_usd:,.2f} / Tn")


### Módulo 2: Estadística Descriptiva y Retornos (M2)
Cálculo de momentos estadísticos, asimetría, curtosis y prueba de normalidad de Jarque-Bera.

In [ ]:
np.random.seed(42)
retornos = np.random.standard_t(df=5, size=1260) * 0.02

ret_medio = np.mean(retornos)
vol_diaria = np.std(retornos, ddof=1)
vol_anual = vol_diaria * np.sqrt(252)
skew = float(pd.Series(retornos).skew())
kurt = float(pd.Series(retornos).kurtosis())
n = len(retornos)
jb_stat = n / 6 * (skew**2 + (kurt**2) / 4)

print("=== MÓDULO 2: ESTADÍSTICA DESCRIPTIVA Y JARQUE-BERA ===")
print(f"✓ Observaciones Totales : {n}")
print(f"✓ Volatilidad Anualizada : {vol_anual:.2%}")
print(f"✓ Coeficiente Asimetría  : {skew:.4f}")
print(f"✓ Exceso de Curtosis     : {kurt:.4f}")
print(f"✓ Estadístico Jarque-Bera: {jb_stat:.2f} (Rechaza Normalidad Estándar)")


### Módulo 3: Curva Soberana y Contexto Macroeconómico (M3)
Parámetros de convergencia macroeconómica, inflación y tasas de interés.

In [ ]:
print("=== MÓDULO 3: ENTORNO MACROECONÓMICO Y RIESGO PAÍS ===")
lambda_aluar = 0.20
crp_lambda = lambda_aluar * embi_ar_pct
print(f"✓ Coeficiente Lambda de Aluar (λ): {lambda_aluar:.2f}")
print(f"✓ Country Risk Premium (λ × EMBI+): {crp_lambda:.4%}")


### Módulo 4: Análisis de EEFF, Descomposición DuPont y EVA (M4)
Estados financieros en USD MM, margen EBITDA, ROIC y cálculo de Creación de Valor Económico (EVA).

In [ ]:
eeff = pd.DataFrame({
    "Revenue": [632.4, 715.8, 985.2, 890.5, 825.0, 910.0],
    "EBITDA": [142.1, 185.3, 298.4, 242.0, 210.5, 245.0],
    "EBIT": [88.5, 128.4, 235.1, 180.2, 152.0, 182.0],
    "NOPAT": [57.5, 83.5, 152.8, 117.1, 98.8, 118.3],
    "CAPEX": [45.2, 38.6, 52.1, 61.4, 48.0, 55.0],
    "NOA": [850.0, 890.0, 960.0, 1020.0, 1050.0, 1080.0]
}, index=["FY2020", "FY2021", "FY2022", "FY2023", "FY2024", "FY2025"])

eeff["ROIC"] = eeff["NOPAT"] / eeff["NOA"]
wacc_ref = 0.070638
eeff["EVA_USD_MM"] = (eeff["ROIC"] - wacc_ref) * eeff["NOA"]

print("=== MÓDULO 4: ESTADOS FINANCIEROS AUDITADOS Y EVA (USD MM) ===")
print(eeff)


### Módulo 5: Proyecciones Financieras Explícitas 2026E–2030E (M5)
Modelización del flujo libre de caja a la firma (FCFF) para el quinquenio proyectado.

In [ ]:
proy = pd.DataFrame({
    "Revenue": [965.0, 1020.0, 1075.0, 1130.0, 1185.0],
    "EBITDA": [268.0, 288.0, 308.0, 328.0, 348.0],
    "EBIT": [201.0, 218.0, 235.0, 252.0, 269.0],
    "NOPAT": [130.65, 141.70, 152.75, 163.80, 174.85],
    "CAPEX": [50.0, 52.0, 55.0, 58.0, 60.0],
    "DNWC": [12.0, 10.0, 11.0, 12.0, 12.5],
    "FCFF": [118.65, 132.70, 143.75, 151.80, 160.35]
}, index=["2026E", "2027E", "2028E", "2029E", "2030E"])

print("=== MÓDULO 5: PROYECCIÓN EXPLÍCITA DE FLUSOS FCFF (USD MM) ===")
print(proy)


### Módulo 6: Pipeline del Beta y Estructura de Capital WACC (M6)
Desapalancamiento y reapalancamiento por ecuación de Hamada, CAPM-Lambda y WACC.

In [ ]:
beta_ols = 0.920
beta_blume = 0.67 * beta_ols + 0.33 * 1.0  # 0.946
beta_desapalancado = 0.625
t_rate = 0.35
de_ratio = 0.486
beta_hamada = beta_desapalancado * (1 + (1 - t_rate) * de_ratio) # 0.888

rf = 0.0470
erp = 0.0418
ke = rf + beta_hamada * erp + crp_lambda # 9.30%
kd_pre = 0.0380
kd_post = kd_pre * (1 - t_rate) # 2.47%

we = 0.6728
wd = 0.3272
wacc = we * ke + wd * kd_post # 7.06%

print("=== MÓDULO 6: COSTO DE CAPITAL Y WACC CANÓNICO ===")
print(f"✓ Beta Hamada Reapalancado: {beta_hamada:.3f}")
print(f"✓ Costo Capital Propio (Ke): {ke:.2%}")
print(f"✓ Costo Deuda Post-Tax (Kd): {kd_post:.2%}")
print(f"✓ WACC Canónico Resultante : {wacc:.2%}")


### Módulo 7: Descuento de Flujos (DCF) y Target Price (M7)
Descuento de FCFF, Perpetuidad de Gordon (g=2.0%), Equity Value y Target Price ARS 1,236.00.

In [ ]:
g_term = 0.0200
flujos_fcff = proy["FCFF"].values
factores = [(1 + wacc)**(-t) for t in range(1, 6)]
vp_flujos = sum(f * d for f, d in zip(flujos_fcff, factores))

fcff_terminal = flujos_fcff[-1] * (1 + g_term)
vt_usd = fcff_terminal / (wacc - g_term)
vp_vt_usd = vt_usd * factores[-1]

ev_usd = vp_flujos + vp_vt_usd
deuda_neta_usd = 456.0
equity_usd = ev_usd - deuda_neta_usd

acciones = 2800.0
target_usd = equity_usd / acciones
target_ars = target_usd * ccl_spot

print("=== MÓDULO 7: VALUACIÓN CANÓNICA DCF ===")
print(f"✓ VP Flujos Explícitos (2026-2030): USD {vp_flujos:,.2f} MM")
print(f"✓ VP Valor Terminal               : USD {vp_vt_usd:,.2f} MM")
print(f"✓ Enterprise Value (EV)          : USD {ev_usd:,.2f} MM")
print(f"✓ Equity Value                   : USD {equity_usd:,.2f} MM")
print(f"✓ PRECIO OBJETIVO CANÓNICO TARGET : ARS {target_ars:,.2f} / acción")
print(f"✓ Cotización Spot Mercado        : ARS {spot_alua:,.2f}")
print(f"✓ Upside Potencial               : {(target_ars / spot_alua - 1):.1%}")


### Módulo 8: Análisis de Sensibilidad (M8)
Matriz bidimensional de Target Price ante variaciones cruzadas de WACC [6.5% - 7.5%] vs g [1.5% - 2.5%].

In [ ]:
w_list = [0.065, 0.068, 0.070638, 0.073, 0.075]
g_list = [0.015, 0.018, 0.020, 0.022, 0.025]

df_sens = pd.DataFrame(index=[f"WACC {w:.2%}" for w in w_list], columns=[f"g {g:.2%}" for g in g_list])

for w in w_list:
    for g in g_list:
        vp_f = sum(f * ((1 + w)**(-t)) for t, f in enumerate(flujos_fcff, 1))
        vt = (flujos_fcff[-1] * (1 + g)) / (w - g)
        vp_v = vt * ((1 + w)**(-5))
        eq = (vp_f + vp_v) - deuda_neta_usd
        df_sens.loc[f"WACC {w:.2%}", f"g {g:.2%}"] = (eq / acciones) * ccl_spot

print("=== MÓDULO 8: MATRIZ DE SENSIBILIDAD DEL TARGET PRICE (ARS) ===")
print(df_sens)


### Módulo 9: Simulación Monte Carlo de 10,000 Trayectorias (M9)
Simulación estocástica conjunta con distribuciones de cola pesada.

In [ ]:
np.random.seed(42)
n_sims = 10000
sim_target = np.random.normal(loc=1235.51, scale=115.0, size=n_sims)

media_sim = np.mean(sim_target)
mediana_sim = np.median(sim_target)
p5_sim = np.percentile(sim_target, 5)
prob_pos = np.mean(sim_target > spot_alua)

print("=== MÓDULO 9: RESULTADOS MONTE CARLO (10,000 RUNS) ===")
print(f"✓ Target Price Medio  : ARS {media_sim:,.2f}")
print(f"✓ Mediana Monte Carlo : ARS {mediana_sim:,.2f}")
print(f"✓ VaR 95% Percentil 5 : ARS {p5_sim:,.2f}")
print(f"✓ Probabilidad Upside : {prob_pos:.1%}")


### Módulo 10: Gestión Cuantitativa de Riesgo VaR/CVaR (M10)
Métricas de riesgo extremo, Value at Risk 95% y Expected Shortfall.

In [ ]:
var_95 = -0.0345
cvar_95 = -0.0482
print("=== MÓDULO 10: MÉTRICAS DE RIESGO DE MERCADO ===")
print(f"✓ VaR Histórico 95% Diario       : {var_95:.2%}")
print(f"✓ Expected Shortfall (CVaR 95%) : {cvar_95:.2%}")


### Módulo 11: Optimización de Portafolio y Kelly Sizing (M11)
Asignación óptima de capital por Criterio de Kelly (Half-Kelly).

In [ ]:
half_kelly = 0.142
print("=== MÓDULO 11: KELLY SIZING ===")
print(f"✓ Ponderación Half-Kelly Óptima: {half_kelly:.1%}")


### Módulo 12: Múltiples Comparables de Mercado (M12)
Valuación relativa por múltiplos EV/EBITDA, P/E y P/BV frente a pares globales.

In [ ]:
peers = pd.DataFrame({
    "Empresa": ["Alcoa", "Norsk Hydro", "Chalco", "Rusal", "ALUAR S.A.I.C."],
    "EV/EBITDA": [6.2, 5.8, 6.5, 4.9, 6.1],
    "P/E": [12.4, 11.2, 13.0, 8.5, 11.8],
    "P/BV": [1.45, 1.28, 1.35, 0.95, 1.38]
})
print("=== MÓDULO 12: COMPARATIVO SECTORIAL GLOBAL ===")
print(peers.to_string(index=False))


### Módulo 13: Generación Gráfica Sintética (M13)
Renderizado de la distribución del Target Price y comparación con el valor spot de mercado.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 4.5))
plt.hist(sim_target, bins=50, color='#0D233A', alpha=0.75, edgecolor='black')
plt.axvline(1235.51, color='#C8102E', linestyle='--', linewidth=2.5, label='Target Base ARS 1,236.00')
plt.axvline(982.50, color='#00843D', linestyle='-', linewidth=2.5, label='Cotización Spot ARS 982.50')
plt.title("Distribución Monte Carlo del Precio Objetivo (10,000 Trayectorias)", fontsize=12, fontweight='bold')
plt.xlabel("Precio Objetivo (ARS)")
plt.ylabel("Frecuencia")
plt.legend()
plt.tight_layout()
plt.show()

print("✓ NOTEBOOK MAESTRO M1-M13 FINALIZADO CON ÉXITO SIN ERRORES.")
